In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

In [2]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
train_dataset = ImageFolder(
    root="/content/drive/MyDrive/zoomcamp_homework/hw8/data/train",
    transform=train_transform,
)

evaluation_dataset = ImageFolder(
    root="/content/drive/MyDrive/zoomcamp_homework/hw8/data/test",
    transform=evaluation_transform,
)

In [6]:
print("Train:", len(train_dataset))
print("Evaluation:", len(evaluation_dataset))

print("Classes:", train_dataset.class_to_idx)

Train: 800
Evaluation: 201
Classes: {'curly': 0, 'straight': 1}


In [7]:
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [8]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("First 10 labels:", labels[:10])

Images shape: torch.Size([20, 3, 200, 200])
Labels shape: torch.Size([20])
First 10 labels: tensor([1, 1, 1, 0, 0, 1, 0, 1, 1, 0])


In [9]:
import torch.nn as nn

In [10]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=0,
        )

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.hidden = nn.Linear(32 * 99 * 99, 64)

        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))

        x = x.flatten(start_dim=1)

        x = nn.functional.relu(self.hidden(x))

        return self.output(x)

In [11]:
model = HairModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

In [12]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Total parameters:", total_params)

Total parameters: 20073473


In [13]:
train_loss = []
train_accuracy = []

evaluation_loss = []
evaluation_accuracy = []

In [14]:
model.train()

HairModel(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

In [15]:
train_loss = []
train_accuracy = []

evaluation_loss = []
evaluation_accuracy = []

for epoch in range(10):
    print(f"Epoch {epoch + 1}/10")

    model.train()

    epoch_loss = 0
    epoch_correct = 0
    epoch_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        logits = model(images)

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(logits) >= 0.5)

        epoch_correct += (predictions == labels).sum().item()
        epoch_total += labels.size(0)

    avg_train_loss = epoch_loss / epoch_total
    avg_train_accuracy = epoch_correct / epoch_total

    train_loss.append(avg_train_loss)
    train_accuracy.append(avg_train_accuracy)

    print(f"train_loss: {avg_train_loss:.4f}, train_accuracy: {avg_train_accuracy:.4f}")

    model.eval()

    eval_loss = 0
    eval_correct = 0
    eval_total = 0

    with torch.no_grad():
        for images, labels in evaluation_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            logits = model(images)

            loss = criterion(logits, labels)

            eval_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(logits) >= 0.5)

            eval_correct += (predictions == labels).sum().item()
            eval_total += labels.size(0)

    avg_eval_loss = eval_loss / eval_total
    avg_eval_accuracy = eval_correct / eval_total

    evaluation_loss.append(avg_eval_loss)
    evaluation_accuracy.append(avg_eval_accuracy)

    print(f"evaluation_loss: {avg_eval_loss:.4f}, evaluation_accuracy: {avg_eval_accuracy:.4f}")

Epoch 1/10


KeyboardInterrupt: 

In [16]:
import json

history = {
    "train_loss": train_loss,
    "train_accuracy": train_accuracy,
    "evaluation_loss": evaluation_loss,
    "evaluation_accuracy": evaluation_accuracy
}

with open("history_baseline.json", "w") as f:
    json.dump(history, f)

print("History saved!")

History saved!


In [19]:
import json
import numpy as np

path = "/content/drive/MyDrive/zoomcamp_homework/hw8/history_baseline.json"

with open(path, "r") as f:
    history = json.load(f)

print("Question 3 — Median train_accuracy:",
      round(np.median(history["train_accuracy"]), 2))

print("Question 4 — Std train_loss:",
      round(np.std(history["train_loss"], ddof=0), 3))

Question 3 — Median train_accuracy: 0.86
Question 4 — Std train_loss: 0.13


In [20]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),

    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),

    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [22]:
train_dataset_augmented = ImageFolder(
    root="/content/drive/MyDrive/zoomcamp_homework/hw8/data/train",
    transform=train_transform_augmented
)

train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=32,
    shuffle=True
)